# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a practical walkthrough for loading, inspecting, and analyzing a Croissant-formatted dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described by a Croissant schema and available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata and Croissant structure
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as an object
md = dataset.metadata

print(f"Dataset Name: {md.name}\n\nDescription: {md.description}")

## 2. Data Overview
Let's review available record sets, fields, and their `@id`s as described by the Croissant schema.

In [ ]:
# List all record sets and their fields with @ids

# Helper: get all record sets defined in the dataset schema
record_sets = list(dataset.record_sets)
print(f"Available record sets ({len(record_sets)}):\n")
for record_set in record_sets:
    print(f"- @id: {record_set['@id']}")
    print(f"  name: {record_set.get('name', '(unnamed)')}")
    # Show fields
    if 'field' in record_set:
        fields = record_set['field']
        # fields can be a list or a dict if length 1
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            print(f"    - @id: {field['@id']} (dataType: {field.get('dataType')})")
    print()

## 3. Data Extraction
Let's load data from specific record sets into pandas DataFrames for analysis. All record sets and field references are handled via their `@id`.

In [ ]:
# Extract data from each record set, using their @id

available_record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Record set @ids:", available_record_set_ids)

# Load all records from all record sets into dataframes
dataframes = {}
for record_set_id in available_record_set_ids:
    # dataset.records yields dictionaries with field @id as keys
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        continue  # May be empty
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set {record_set_id}")

# Choose the first non-empty record set for initial exploration
if dataframes:
    primary_record_set_id = next(iter(dataframes.keys()))
    print(f"\nPrimary record set selected: {primary_record_set_id}")
    print(f"Fields available in {primary_record_set_id} (column names are field @ids):")
    print(list(dataframes[primary_record_set_id].columns))
    display(dataframes[primary_record_set_id].head(3))
else:
    print("No data available in any record set.")

## 4. Exploratory Data Analysis (EDA)
We will examine a numeric field, apply a filter to remove outliers, normalize the data, and optionally group by a categorical field if present. All field references are by their `@id` as per the schema.

In [ ]:
# EDA: filter/norm/group, using field @ids only
import numpy as np

if dataframes:
    df = dataframes[primary_record_set_id]
    # Identify a numeric field by inferring dtype (float/int); use the first available
    numeric_field_id = None
    for col in df.columns:
        # Try to infer numeric columns by checking dtype
        if np.issubdtype(df[col].dropna().apply(lambda x: type(x).__name__).mode()[0], str):
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                continue
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
    if numeric_field_id is not None:
        print(f"Selected numeric field for analysis: {numeric_field_id}")
        # Set filter threshold at 75th percentile to remove potential outliers
        threshold = df[numeric_field_id].quantile(0.75)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {round(threshold,2)}:")
        display(filtered_df.head(3))
        # Normalize
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
        print(f\"Normalized values of {numeric_field_id} (z-score):\")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))
        # Optionally group by a categorical field (not the numeric)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < df.shape[0] // 2:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head(3))
    else:
        print("No numeric field detected for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Let's visualize the distribution of the numeric field and its relationship to the group (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Visualize boxplot by group if group field available
    if group_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated:

- Accessing and exploring a Croissant-annotated dataset using the `mlcroissant` library
- Listing record sets and fields using their `@id` as unique identifiers
- Extracting, filtering, normalizing, and grouping data using field and record set `@id`s
- Visualizing distributions and category-wise summaries

This approach ensures data provenance and schema-driven clarity in analysis workflows. For more advanced analyses, see the `mlcroissant` documentation.